In [1]:
from PIL import Image
from transformers import AutoProcessor, AutoModelForImageTextToText
import torch


c:\Users\fomka\anaconda3\envs\llm_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
model_path = "PaddlePaddle/PaddleOCR-VL-1.5"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
model = AutoModelForImageTextToText.from_pretrained(model_path,
                                                   trust_remote_code=True,
                                                   dtype=torch.bfloat16,
                                                   text_config='').to(DEVICE).eval()
processor = AutoProcessor.from_pretrained(model_path)

Unrecognized keys in `rope_parameters` for 'rope_type'='default': {'mrope_section'}


TypeError: PaddleOCRVLForConditionalGeneration.__init__() got an unexpected keyword argument 'text_config'

In [ ]:
from pathlib import Path

ds_path = Path('dataset')
image_path = str(ds_path / '_Distributed Cloud Computing. Big Data processing methods_ workshop.jpg')
image = Image.open(image_path).convert("RGB")
task = "ocr"
orig_w, orig_h = image.size
spotting_upscale_threshold = 1500

PROMPTS = {
    "ocr": "OCR:",
    "table": "Table Recognition:",
    "formula": "Formula Recognition:",
    "chart": "Chart Recognition:",
    "spotting": "Spotting:",
    "seal": "Seal Recognition:",
}


In [ ]:
if task == "spotting" and orig_w < spotting_upscale_threshold and orig_h < spotting_upscale_threshold:
    process_w, process_h = orig_w * 2, orig_h * 2
    try:
        resample_filter = Image.Resampling.LANCZOS
    except AttributeError:
        resample_filter = Image.LANCZOS
    image = image.resize((process_w, process_h), resample_filter)

# Set max_pixels: use 1605632 for spotting, otherwise use default ~1M pixels
max_pixels = 2048 * 28 * 28 if task == "spotting" else 1280 * 28 * 28

In [ ]:
model = AutoModelForImageTextToText.from_pretrained(model_path, torch_dtype=torch.bfloat16).to(DEVICE).eval()
processor = AutoProcessor.from_pretrained(model_path)

In [ ]:
messages = [
    {
        "role": "user",
        "content": [
            {"type": "image", "image": image},
            {"type": "text", "text": PROMPTS[task]},
        ]
    }
]
inputs = processor.apply_chat_template(
    messages,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt",
    images_kwargs={"size": {"shortest_edge": processor.image_processor.min_pixels, "longest_edge": max_pixels}},
).to(model.device)

In [ ]:
inputs

In [ ]:
outputs = model.generate(**inputs, max_new_tokens=512)

In [ ]:
result = processor.decode(outputs[0][inputs["input_ids"].shape[-1]:-1])
print(result)

In [ ]:
import json
from datetime import datetime
from pathlib import Path
from tqdm import tqdm
import io

# Попытка импорта библиотеки для работы с PDF
try:
    import fitz  # PyMuPDF
    PDF_SUPPORT = True
except ImportError:
    PDF_SUPPORT = False
    print("⚠️ Библиотека PyMuPDF не найдена. PDF-файлы будут пропущены.")
    print("   Установите: pip install PyMuPDF")

# --- Конфигурация ---
DATASET_DIR = Path('dataset')
OUTPUT_FILE = 'ocr_results.json'
IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.tiff', '.webp'}
PDF_EXTENSIONS = {'.pdf'}

# --- Загрузка существующих результатов (если есть) ---
if Path(OUTPUT_FILE).exists():
    with open(OUTPUT_FILE, 'r', encoding='utf-8') as f:
        results = json.load(f)
    processed_files = {item['file_name'] for item in results.get('data', [])}
    print(f"📂 Найден существующий файл {OUTPUT_FILE}")
    print(f"📋 Уже обработано файлов: {len(processed_files)}")
else:
    results = {
        "extraction_date": datetime.now().isoformat(),
        "data": []
    }
    processed_files = set()
    print(f"📄 Файл {OUTPUT_FILE} не найден, создаем новый")

# --- Поиск всех изображений и PDF в папке ---
all_files = [
    f for f in DATASET_DIR.iterdir()
    if f.is_file() and (f.suffix.lower() in IMAGE_EXTENSIONS or f.suffix.lower() in PDF_EXTENSIONS)
]

# --- Фильтрация: оставляем только новые файлы ---
new_files = [f for f in all_files if f.name not in processed_files][:500]
skipped_count = len(all_files) - len(new_files)

image_count = sum(1 for f in new_files if f.suffix.lower() in IMAGE_EXTENSIONS)
pdf_count = sum(1 for f in new_files if f.suffix.lower() in PDF_EXTENSIONS)

print(f"📁 Всего файлов в папке: {len(all_files)}")
print(f"⏭️ Пропущено (уже обработано): {skipped_count}")
print(f"🆕 Новые файлы для обработки: {len(new_files)}")
print(f"   ├─ Изображения: {image_count}")
print(f"   └─ PDF: {pdf_count}")

# --- Функция загрузки изображения (поддержка JPG/PNG и PDF) ---
def load_image_from_file(file_path):
    """Загружает изображение из файла (поддерживает изображения и PDF)"""
    suffix = file_path.suffix.lower()

    if suffix in IMAGE_EXTENSIONS:
        return Image.open(file_path).convert("RGB")

    elif suffix in PDF_EXTENSIONS:
        if not PDF_SUPPORT:
            raise ImportError("PyMuPDF не установлен для обработки PDF")

        # Открываем PDF и конвертируем первую страницу в изображение
        doc = fitz.open(file_path)
        page = doc[0]  # Первая страница

        # Рендерим страницу в изображение с высоким разрешением (300 DPI)
        mat = fitz.Matrix(300/72, 300/72)  # Увеличиваем разрешение для лучшего OCR
        pix = page.get_pixmap(matrix=mat)

        # Конвертируем в PIL Image
        img_data = pix.tobytes("png")
        image = Image.open(io.BytesIO(img_data)).convert("RGB")

        doc.close()
        return image

    else:
        raise ValueError(f"Неподдерживаемый формат файла: {suffix}")

# --- Обработка новых файлов ---
for file_path in tqdm(new_files, desc="Обработка сертификатов"):
    try:
        # Загрузка и подготовка изображения
        image = load_image_from_file(file_path)
        orig_w, orig_h = image.size

        # Логика масштабирования (из вашего кода)
        spotting_upscale_threshold = 1500
        if task == "spotting" and orig_w < spotting_upscale_threshold and orig_h < spotting_upscale_threshold:
            try:
                resample_filter = Image.Resampling.LANCZOS
            except AttributeError:
                resample_filter = Image.LANCZOS
            image = image.resize((orig_w * 2, orig_h * 2), resample_filter)

        # Формирование запроса к модели
        messages = [
            {
                "role": "user",
                "content": [
                    {"type": "image", "image": image},
                    {"type": "text", "text": PROMPTS[task]},
                ]
            }
        ]

        inputs = processor.apply_chat_template(
            messages,
            add_generation_prompt=True,
            tokenize=True,
            return_dict=True,
            return_tensors="pt",
            images_kwargs={"size": {"shortest_edge": processor.image_processor.min_pixels, "longest_edge": max_pixels}},
        ).to(model.device)

        # Генерация текста
        outputs = model.generate(**inputs, max_new_tokens=512)
        result_text = processor.decode(outputs[0][inputs["input_ids"].shape[-1]:-1])

        # Сохранение результата
        results["data"].append({
            "file_name": file_path.name,
            "result": result_text.strip()
        })

        # Сохраняем промежуточные результаты после каждого файла
        with open(OUTPUT_FILE, 'w', encoding='utf-8') as f:
            json.dump(results, f, ensure_ascii=False, indent=2)

    except Exception as e:
        print(f"❌ Ошибка обработки файла {file_path.name}: {e}")
        results["data"].append({
            "file_name": file_path.name,
            "result": f"ERROR: {str(e)}"
        })
        # Сохраняем даже при ошибке
        with open(OUTPUT_FILE, 'w', encoding='utf-8') as f:
            json.dump(results, f, ensure_ascii=False, indent=2)

# --- Финальное сохранение ---
with open(OUTPUT_FILE, 'w', encoding='utf-8') as f:
    json.dump(results, f, ensure_ascii=False, indent=2)

print(f"\n✅ Обработка завершена!")
print(f"📊 Всего обработано файлов в этот раз: {len(new_files)}")
print(f"📈 Всего файлов в результатах: {len(results['data'])}")
print(f"💾 Результаты сохранены в {OUTPUT_FILE}")

In [ ]:
from paddleocr import PaddleOCRVL

In [ ]:
pipeline = PaddleOCRVL()